In [44]:
from concurrent.futures import ThreadPoolExecutor
import time
import requests
import pandas as pd
import re
import psycopg2

In [2]:
df= pd.read_csv('Employer Information (1).csv', encoding='utf-16', sep='\t', low_memory=False)
df.head(10)

,Line by line,Fiscal Year,Employer (Petitioner) Name,Tax ID,Industry (NAICS) Code,Petitioner City,Petitioner State,Petitioner Zip Code,New Employment Approval,New Employment Denial,Continuation Approval,Continuation Denial,Change with Same Employer Approval,Change with Same Employer Denial,New Concurrent Approval,New Concurrent Denial,Change of Employer Approval,Change of Employer Denial,Amended Approval,Amended Denial
0,1,2026,NaN,3227.0,"54 - Professional, Scientific, and Technical S...",CANTON,MI,48187.0,0,0,0,0,0,0,0,0,0,0,0,1
1,2,2026,NaN,4902.0,31-33 - Manufacturing,RANDOLPH,MA,2368.0,0,0,1,0,0,0,0,0,0,0,0,0
2,3,2026,NaN,9012.0,31-33 - Manufacturing,HILLSBOROUGH,NH,3244.0,0,0,1,0,0,0,0,0,0,0,0,0
3,4,2026,1 ALPHA TECHNOLOGIES LLC,4511.0,"54 - Professional, Scientific, and Technical S...",HOUSTON,TX,77098.0,0,0,0,0,0,0,0,0,0,1,0,0
4,5,2026,1 DUST GROUP LLC,5330.0,"54 - Professional, Scientific, and Technical S...",FARMERS BRANCH,TX,75234.0,1,0,0,0,0,0,0,0,0,0,0,0
5,6,2026,1 EMC LLC,9142.0,72 - Accommodation and Food Services,FINDLAY,OH,45840.0,1,0,0,0,0,0,0,0,0,0,0,0
6,7,2026,1 NET SOFTWARE TECHNOLOGIES,9485.0,"54 - Professional, Scientific, and Technical S...",IRVING,TX,75063.0,0,0,0,0,0,0,0,0,0,0,1,0
7,8,2026,"1 WAY SOLUTIONS, INC.",3700.0,"54 - Professional, Scientific, and Technical S...",ATLANTA,GA,30062.0,0,0,0,0,0,0,0,0,1,0,0,0
8,9,2026,1-800-FLOWERS COM INC,7311.0,44-45 - Retail Trade,JERICHO,NY,11753.0,0,0,1,0,0,0,0,0,0,0,0,0
9,10,2026,1FINITY AMERICAS INC,4286.0,"54 - Professional, Scientific, and Technical S...",DALLAS,TX,75252.0,1,0,2,0,0,0,0,0,1,0,0,0


In [8]:
unique_companies = df['Employer (Petitioner) Name'].dropna().unique()
print(len(unique_companies))
type(unique_companies)

69752


pandas.arrays.StringArray

In [39]:
import re

def cleaning_name(name):
    name = re.sub(r'\b(llc|ltd|inc|corp|co|lp|llp|pvt|limited|corporation|)\b','',name,flags=re.IGNORECASE).lower()

    name = re.sub(r'[^a-z0-9]', '', name) 
    return name

def get_data_jobs(slug):
    # slug = cleaning_name(slug)
    url = f"https://boards-api.greenhouse.io/v1/boards/{slug}/jobs"
    res= requests.get(url)
    us_keywords = ['usa', 'united states', 'united states of america', 'remote']


    if res.status_code!=200:
        return []
    jobs_list = res.json()['jobs']
    matches=[]
    for i in jobs_list:
        if 'data' in i['title'].lower() and any(kw in i['location']['name'].lower() for kw in us_keywords):
            matches.append({'company':slug,
                            'title':i['title'],
                            'location':i['location']['name'],
                            'url': i['absolute_url']

            
                            })
    return matches
    


In [9]:
cleaned_slugs = list(set([cleaning_name(name) for name in unique_companies]))
print(len(cleaned_slugs))

66673


In [28]:
def check_slug(slug):
    if not slug or len(slug) < 2:  # skip empty or too-short slugs
        return None
    url = f"https://boards-api.greenhouse.io/v1/boards/{slug}/jobs"
    res = requests.get(url)
    for attempt in range(3):  # loop runs max 3 times
        try:
            res = requests.get(url, timeout=2)
            if res.status_code == 200:
                return slug   # ← exits immediately, loop stops
            return None       # ← exits immediately, loop stops
        except Exception as e:
            #print(f"Error occurred while checking slug '{slug}': {e}")
            time.sleep(1)     # ← only reaches here if request CRASHED




In [43]:
start = time.time()
#test_slugs = cleaned_slugs[:7500]
with ThreadPoolExecutor(max_workers=10) as executor:
    results = list(executor.map(check_slug, cleaned_slugs))

confirmed = [r for r in results if r is not None]

end = time.time()
print(f"Time: {end-start:.2f} seconds")
print(f"Found on Greenhouse: {len(confirmed)}")
print(confirmed)

ConnectionError: HTTPSConnectionPool(host='boards-api.greenhouse.io', port=443): Max retries exceeded with url: /v1/boards/inpatientconsultantsofnevadaamedical/jobs (Caused by NameResolutionError("HTTPSConnection(host='boards-api.greenhouse.io', port=443): Failed to resolve 'boards-api.greenhouse.io' ([Errno -3] Temporary failure in name resolution)"))

In [ ]:
pd.Series(confirmed).to_csv('confirmed_greenhouse_slugs.csv', index=False)
print("saved!")


saved!
